# 第6回：教師なし学習基礎 〜 データの「隠れた構造」を発掘する 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session06_unsupervised_learning.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

1. **【講義1】教師なし学習の哲学 (10分)**
   - 「正解」がない中で AI は何を見ているのか？ クラスタリングと次元圧縮。
2. **【講義2】K-means 法：自動チーム分けのアルゴリズム (15分)**
   - 重心の更新。エルボー法による最適なクラスター数 $K$ の決定。
3. **【実習1】未知の疾患サブタイプを見つける (15分)**
   - 合成患者データを用いたクラスタリング実践。
4. **【講義3】主成分分析 (PCA)：情報の要約術 (15分)**
   - 分散の最大化。寄与率と累積寄与率。主成分負荷量（何が効いているか）。
5. **【実習2】高次元データの 2 次元圧縮と可視化 (15分)**
   - 10個の検査数値を 2 つの軸に凝縮する。
6. **【講義4】階層的クラスタリング：データの間柄 (10分)**
   - デンドログラム（系統樹）の読み方と作り方。
7. **【総合演習】「バイオマーカー発見」シミュレーション (10分)**

---

## 1. セットアップ

高度な次元圧縮（UMAP 等）やインタラクティブな描画、系統樹作成ライブラリを使用します。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style='white', palette='husl')

print("探索的データマイニングエンジン、起動。")

## 2. 【講義1+2】クラスタリング：K-means の数理

### 2.1 アルゴリズムの真実
K-means は、各データ点から一番近い「重心 (Centroid)」までの距離の 2 乗和を最小にするように動きます。重心を動かす $\leftrightarrow$ 所属チームを決める、を交互に繰り返すだけのシンプルなルールですが、強力です。

### 2.2 エルボー法：いくつに分けるべきか？
クラスター数 $K$ を増やすほど、中のバラツキは減りますが、$K$ を増やしすぎると意味がなくなります。バラツキの減り方が急激に緩やかになる場所（肘：Elbow）を最適な数と判断します。

---

### 【実習1】K-means による疾患群の自動発見

In [ ]:
from sklearn.datasets import make_blobs

# 擬似的な 4 つのサブタイプを持つ患者データ (300人, 2項目: 血圧, 血糖)
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# クラスタリング実行 (K=4)
kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(X)

# 視覚化
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=60, alpha=0.7, edgecolor='w')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            marker='X', s=200, c='red', label='Centroids')
plt.title("Discovery of Unknown Patient Subtypes")
plt.legend()
plt.show()

---

## 3. 【講義3】次元圧縮：PCA (主成分分析)

### 3.1 情報の「影」を一番長く伸ばす
主成分分析は、データの「最もバラツキが大きい方向」を探す手法です。10個の検査項目があった時、それらを合成して「第1主成分」という新しい最強の指標を 1 つ作ります。

### 3.2 解釈の鍵：負荷量 (Loadings)
- **第1主成分の正体**: 「年齢と血圧と血糖値が全部高い方向」かもしれません。これは「老化・代謝リスク」というメタ・指標と言えます。これを数値で見ることができるのが負荷量です。

---

### 【実習2】多次元検査データの PCA 圧縮

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
X_iris = iris.data

# 教師なし学習では「標準化」が命！
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# PCA 実行
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"第1主成分の寄与率: {pca.explained_variance_ratio_[0]:.4f}")
print(f"第2主成分の寄与率: {pca.explained_variance_ratio_[1]:.4f}")
print(f"合計で情報の {np.sum(pca.explained_variance_ratio_)*100:.1f}% を保持しています。")

# 負荷量を確認（どの特徴が効いているか）
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=iris.feature_names)
display(loadings)

## 4. 【講義4】デンドログラム：関係性の家系図

「個体 A と B は似ている」「C は少し離れている」という関係を、トーナメント表（樹形図）にするのが **階層的クラスタリング** です。K-means のように最初にチーム数を決めなくて良いのが利点です。

---

### 【実習3】デンドログラムの描画

In [ ]:
# データの類似度を計算して結合
linked = linkage(X_scaled[:30], 'ward') # 表示が重くなるため最初の 30 件のみ

plt.figure(figsize=(10, 7))
dendrogram(linked, labels=iris.target[:30])
plt.title("Dendrogram of Patient Similarity")
plt.show()

---

## ✏️ 本日の最終研究ミッション (Discovery Challenge)

**シナリオ**: 患者 300 名の 25 項目のオミクスデータ（擬似データ）があります。

### 課題 1: 最適な K の探索
K-means を $K=2$ から $K=10$ まで回し、`kmeans.inertia_`（クラスタ内の分散）をプロットしてエルボー図を作成し、このデータに最適な $K$ はいくつかを提言してください。

### 課題 2: 主成分の意味を解釈せよ
PCA を実行し、「第1主成分」に対して正の負荷量が大きい（寄与が高い）上位 3 つの項目を特定してください。それらの項目から、第1主成分にどのような名前（例：炎症重症度スコア、等）を付けますか？

### 課題 3: 可視化の合わせ技
「PCA で 2 次元に圧縮した後の座標」を使って K-means を実行し、その結果を散布図で表示してください。生データでやるのと何が違うでしょうか？

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)